# Why no planning loop in the 200

Reviewer 4 asks whether F4 is absent because the templates fail to elicit it or because
the backbone is too strong to loop. The two answers call for different follow-up, so
the question is worth settling from the data rather than argued.

The pilot and the main run use the same backbone, the same decoding and the same action
budget, and the pilot contains two planning loops. Whatever separates the corpora is
therefore in the tasks, not in the model. This notebook measures what: the condition
under which the pilot's loops arose is a knowledge-base lookup that returns not-found,
and the question is how often each corpus puts the agent in that position.

In [ ]:
import ast, json, re, urllib.request
from collections import Counter

BASE = ("https://raw.githubusercontent.com/achenna-source/"
        "icv-agent-observability/neucom-d-26-17269-r1/")

def rows(path):
    raw = urllib.request.urlopen(BASE + path, timeout=90).read().decode("utf-8")
    return [json.loads(l) for l in raw.splitlines() if l.strip()]

def steps(r):
    t = r["trajectory"]
    return ast.literal_eval(t) if isinstance(t, str) else t

MAIN = rows("data/trajectories_200.jsonl")
PILOT = rows("data/trajectories_pilot_30.jsonl")
print("main %d, pilot %d" % (len(MAIN), len(PILOT)))
print("labels in the pilot :", Counter(r["label_human"] for r in PILOT).most_common())
print("labels in the main run:", Counter(r["label_human"] for r in MAIN).most_common())

In [ ]:
NOTFOUND = "Information non trouv"          # the knowledge base's not-found return

def notfound_steps(tr):
    return [s for s in tr
            if s.get("action") == "knowledge_base"
            and NOTFOUND in str(s.get("observation") or "")]

def profile(corpus, name):
    n_any = n_two = 0
    tot_steps = 0
    for r in corpus:
        tr = steps(r)
        tot_steps += len(tr)
        k = len(notfound_steps(tr))
        n_any += k > 0
        n_two += k > 1
    print("%-6s %3d trajectories, %.1f steps each" % (name, len(corpus), tot_steps / len(corpus)))
    print("       with at least one not-found return : %3d  (%.1f%%)"
          % (n_any, 100.0 * n_any / len(corpus)))
    print("       with two or more                   : %3d  (%.1f%%)"
          % (n_two, 100.0 * n_two / len(corpus)))
    return n_any, n_two

pa, pt = profile(PILOT, "pilot")
ma, mt = profile(MAIN, "main")

In [ ]:
# the two pilot loops, and what they have in common
print("pilot trajectories labelled F4:")
for r in PILOT:
    if r["label_human"] != "F4":
        continue
    tr = steps(r)
    nf = notfound_steps(tr)
    print("   %s : %d steps, %d not-found returns" % (r["id"], len(tr), len(nf)))
    for s in tr:
        print("      %-16s %-34s -> %s"
              % (s.get("action"), str(s.get("input"))[:34],
                 str(s.get("observation"))[:46]))

In [ ]:
# repeated identical queries, the behaviour a loop is made of, regardless of label
def repeats(tr):
    seen = Counter((s.get("action"), str(s.get("input")).strip().lower()) for s in tr)
    return sum(1 for k, v in seen.items() if v > 1 and k[0] is not None)

for corpus, name in ((PILOT, "pilot"), (MAIN, "main")):
    n = sum(1 for r in corpus if repeats(steps(r)))
    print("%-6s trajectories containing a verbatim repeated call : %3d of %3d (%.1f%%)"
          % (name, n, len(corpus), 100.0 * n / len(corpus)))

print()
print("and among trajectories that did meet a not-found return:")
for corpus, name in ((PILOT, "pilot"), (MAIN, "main")):
    sub = [r for r in corpus if notfound_steps(steps(r))]
    n = sum(1 for r in sub if repeats(steps(r)))
    print("   %-6s %d of %d repeated a call" % (name, n, len(sub)))

In [ ]:
print("=== for the manuscript ===")
print("same backbone, same decoding, same six-action budget in both corpora")
print("pilot: %d of %d trajectories meet a not-found return (%.0f%%), and it holds 2 planning loops"
      % (pa, len(PILOT), 100.0 * pa / len(PILOT)))
print("main : %d of %d (%.0f%%), and it holds none"
      % (ma, len(MAIN), 100.0 * ma / len(MAIN)))

In [ ]:
# the seven main-run trajectories that do repeat a call: what are they labelled, and why not F4
print("main-run trajectories with a verbatim repeated call:")
for r in MAIN:
    tr = steps(r)
    if not repeats(tr):
        continue
    print("   %-6s label %-3s  %d steps" % (r["id"], r["label_human"], len(tr)))
    for s in tr:
        print("      %-16s %-30s -> %s" % (s.get("action"), str(s.get("input"))[:30], str(s.get("observation"))[:40]))

print()
from collections import Counter
lab = Counter(r["label_human"] for r in MAIN if repeats(steps(r)))
print("their labels:", lab.most_common())
print()
print("near-repetition is what the duplicate test measures; verbatim repetition is stricter.")
